### Configure PyTorch CUDA memory allocation settings to prevent fragmentation

In [1]:
 import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

### Check GPU availability and display GPU information

In [2]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)

Mon Mar 16 18:43:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [3]:
 %%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate

### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [4]:
import re
import json
import unicodedata
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

from datasets import load_dataset, Audio, concatenate_datasets
import evaluate

from transformers import (
    Wav2Vec2CTCTokenizer,
    SeamlessM4TFeatureExtractor,
    Wav2Vec2BertProcessor,
    Wav2Vec2BertForCTC,
    TrainingArguments,
    Trainer,
)

### Load the Dhivehi Common Voice dataset and remove unused metadata columns

In [5]:
dv_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
dv_test  = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

dv_train = dv_train.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])
dv_test  = dv_test.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 120379.02it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 128091.07it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 115859.71it/s]


### Install the wget package for downloading files

In [6]:
!pip install wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=6e86776dccac246fc2d7fc5b79c9e13f11c7da1fd07f90c11ee3a9684deb73e2
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


### Load the Sinhala ASR dataset with specified train/test splits

In [7]:
import os
import aiohttp
from datasets import load_dataset

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "180"

splits = load_dataset(
    "keshan/large-sinhala-asr-dataset",
    "si",
    split={
        "train": "train[:40%]",
        "test": "test[:40%]",
    },
    trust_remote_code=True,
    storage_options={
        "client_kwargs": {
            "timeout": aiohttp.ClientTimeout(total=3600)
        }
    },
)

si_train = splits["train"].rename_column("file", "audio").remove_columns(["filename", "x"])
si_test  = splits["test"].rename_column("file", "audio").remove_columns(["filename", "x"])


print("Sinhala train size:", len(si_train))
print("Sinhala test size:", len(si_test))

large-sinhala-asr-dataset.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Sinhala train size: 53018
Sinhala test size: 9356


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [8]:
 # -------- Dhivehi cleaning --------
chars_to_remove_regex_dv = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«]'
arabic_pattern = re.compile(r"[\u0600-\u06FF]")

def dv_clean(batch):
    s = re.sub(chars_to_remove_regex_dv, "", batch["sentence"]).lower()
    # remove Arabic rows
    if arabic_pattern.search(s):
        batch["sentence"] = ""
        return batch
    # remove latin
    s = re.sub(r"[a-z]+", "", s)
    batch["sentence"] = s.strip()
    return batch

# -------- Sinhala cleaning --------
chars_to_remove_regex_si = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\(\)\'\”\‘‘\’\ʻ\”\“\–\—\…\»\«।॥]'
sinhala_allowed = re.compile(r"^[\u0D80-\u0DFF\s]+$")

def si_clean(batch):
    s = unicodedata.normalize("NFC", batch["sentence"])
    s = re.sub(chars_to_remove_regex_si, "", s)
    s = re.sub(r"[A-Za-z0-9]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    batch["sentence"] = s
    return batch

### Apply cleaning functions and filter out empty sentences from both Dhivehi and Sinhala datasets

In [9]:
dv_train = dv_train.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)
dv_test  = dv_test.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)

si_train = si_train.map(si_clean).filter(lambda x: bool(sinhala_allowed.match(x["sentence"])) if x["sentence"] else False)
si_test  = si_test.map(si_clean).filter(lambda x: bool(sinhala_allowed.match(x["sentence"])) if x["sentence"] else False)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

Map:   0%|          | 0/53018 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53018 [00:00<?, ? examples/s]

Map:   0%|          | 0/9356 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9356 [00:00<?, ? examples/s]

### Resample all audio to 16kHz for both Dhivehi and Sinhala datasets

In [10]:
dv_train = dv_train.cast_column("audio", Audio(sampling_rate=16_000))
dv_test  = dv_test.cast_column("audio", Audio(sampling_rate=16_000))
si_train = si_train.cast_column("audio", Audio(sampling_rate=16_000))
si_test  = si_test.cast_column("audio", Audio(sampling_rate=16_000))

### Extract the combined character vocabulary from both Dhivehi and Sinhala datasets

In [11]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

all_train_text = concatenate_datasets([
    dv_train.select_columns(["sentence"]),
    si_train.select_columns(["sentence"]),
])
all_test_text = concatenate_datasets([
    dv_test.select_columns(["sentence"]),
    si_test.select_columns(["sentence"]),
])

vocab_train = all_train_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_train_text.column_names
)
vocab_test = all_test_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_test_text.column_names
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

# Space -> word delimiter |
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]


# Specials
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

print("Vocab size:", len(vocab_dict))

Map:   0%|          | 0/47475 [00:00<?, ? examples/s]

Map:   0%|          | 0/9687 [00:00<?, ? examples/s]

Vocab size: 131


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [12]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [13]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

dv_train_p = dv_train.map(prepare_dataset, remove_columns=dv_train.column_names)
dv_test_p  = dv_test.map(prepare_dataset,  remove_columns=dv_test.column_names)
si_train_p = si_train.map(prepare_dataset, remove_columns=si_train.column_names)
si_test_p  = si_test.map(prepare_dataset,  remove_columns=si_test.column_names)

Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

### Concatenate Dhivehi and Sinhala processed datasets into unified train and test sets

In [14]:
train = concatenate_datasets([dv_train_p, si_train_p]).shuffle(seed=42)
test  = concatenate_datasets([dv_test_p,  si_test_p]).shuffle(seed=42)

print("Train size:", len(train))
print("Test size:", len(test))


Train size: 47475
Test size: 9687


### Define a custom data collator that pads input features and labels for CTC training

In [15]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

### Define the evaluation metrics function computing WER and CER using greedy decoding

In [16]:
import numpy as np

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    pred_ids = np.argmax(pred_logits, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Load the pre-trained W2V-BERT 2.0 model with an adapter and CTC head for fine-tuning

In [17]:
model = Wav2Vec2BertForCTC.from_pretrained(
    "facebook/w2v-bert-2.0",
    add_adapter=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at facebook/w2v-bert-2.0 and are newly initialized: ['lm_head.bias', 'lm_head.weight', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.residual_conv.bias', 'wav2vec2_bert.adapter.layers.0.residual_conv.weight', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.weight', 'wav2vec2_ber

### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [18]:
from transformers import TrainingArguments, Trainer, Wav2Vec2BertForCTC, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="steps",     # FIXED
    eval_steps=300,

    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=9000,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",   # FIXED
    greater_is_better=False,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=processor
)


/tmp/ipykernel_556/996113040.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start the fine-tuning training run

In [19]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 132, 'bos_token_id': 131}.


Step,Training Loss,Validation Loss,Wer,Cer
300,2613.191900,374.403290,0.832975,0.350839
600,885.605600,144.115845,0.567438,0.134554
900,600.618100,117.545395,0.490882,0.109639
1200,450.982000,110.850021,0.475098,0.104342
1500,398.589800,95.984360,0.443529,0.093319
1800,382.504300,91.452225,0.427853,0.090343
2100,331.650300,94.623596,0.439164,0.089042
2400,317.097800,84.558693,0.416348,0.084765
2700,312.552100,83.451759,0.407724,0.081363
3000,279.762800,81.334152,0.389876,0.079126


TrainOutput(global_step=14840, training_loss=302.25981550602256, metrics={'train_runtime': 42645.8247, 'train_samples_per_second': 11.132, 'train_steps_per_second': 0.348, 'total_flos': 6.1457713242918494e+19, 'train_loss': 302.25981550602256, 'epoch': 10.0})

### Mount Google Drive for persistent storage

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Copy the best checkpoint to Google Drive for persistent storage

In [21]:
!cp -r ./outputs/checkpoint-14840 /content/drive/MyDrive/final_model40

### Run greedy CTC inference on the test set and compute WER/CER

In [22]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

all_pred = []
all_refs = []

model.eval()

checkpoint_dir = "./outputs/checkpoint-14840"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

for ex in dv_test_p:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    ref_text = processor.decode(ex["labels"], group_tokens=False)

    all_pred.append(pred_text)
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (Dhivehi only, greedy): {wer:.4f}")
print(f"CER (Dhivehi only, greedy): {cer:.4f}")

for idx in range(min(3, len(all_refs))):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])


WER (Dhivehi only, greedy): 0.4209
CER (Dhivehi only, greedy): 0.0630

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
PRED: ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
PRED: އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
PRED: އެސް ފިޔަތަކުން ވަނީ ނިދިގެއްލިފަ


### Run greedy CTC inference on the test set and compute WER/CER

In [23]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

all_pred = []
all_refs = []

model.eval()

checkpoint_dir = "./outputs/checkpoint-14840"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

for ex in si_test_p:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    ref_text = processor.decode(ex["labels"], group_tokens=False)

    all_pred.append(pred_text)
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (Sinhala only, greedy): {wer:.4f}")
print(f"CER (Sinhala only, greedy): {cer:.4f}")

for idx in range(min(3, len(all_refs))):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])


WER (Sinhala only, greedy): 0.2323
CER (Sinhala only, greedy): 0.0447

--- Example 0 ---
REF : ආචාර්යවරිය නම් ලකුණු කරන අතරතුර
PRED: ආචාර්යවරිය නම් ලකුණු කරන අතරතුර

--- Example 1 ---
REF : මේ අයුරින් මරණයට පත් වේ
PRED: මේ අයුරින් මරණයට පත්වේ

--- Example 2 ---
REF : ඒකට මොකද මේකෙන්
PRED: ඒකට මොකද මේකෙන්
